In [3]:
import numpy as np
import pandas as pd
import msgi

In [4]:
cloud_data = pd.read_parquet("/cluster/work/climate/dnikolo/GTE_validation/DARDAR/Test_bulk/Results/Agg_03_clouds_20070101.0000_20070103.0000.parquet")

In [5]:
cloud_data

,tracknumber,is_large_pix_cloud,is_cot_valid_cloud,is_ctp_valid_cloud,is_liq,is_mix,is_ice,max_water_frac,max_ice_fraction,avg_size[km],...,cth_hist,cth_std_hist,cth_nan_frac_hist,val_intersec_lon,val_intersec_lat,min_temp,max_temp,pole,Hemisphere,Lifetime [h]
0,1,False,False,False,True,False,False,1.000000,0.001011,558296.861966,...,"[1843.9921, 1836.7432, 1849.0636, 1868.0234, 1...","[363.78640281764245, 300.1856456844664, 279.05...","[0.03450292397660819, 0.03220058546519028, 0.0...","[[-58.665966586764824, -58.54676882985961, -58...","[[43.310640296878844, 43.29834658806462, 43.28...",-6,0,np,North,48.00
1,2,False,False,False,True,False,False,1.000000,0.000000,222686.603862,...,"[1322.0084, 1295.5554, 1280.5333, 1261.2963, 1...","[313.45317134302536, 320.5926153235598, 320.62...","[0.04746136865342163, 0.03807303807303807, 0.0...","[[24.151652176569048, 24.220510179531143, 24.2...","[[59.36936222316085, 59.3747313930258, 59.3801...",-6,0,np,North,7.50
2,3,False,False,False,True,False,False,1.000000,0.001748,129286.282023,...,"[2420.172, 2389.1594, 2511.0806, 2496.6367, 24...","[442.15357201994874, 416.3479802100642, 428.93...","[0.037925925925925925, 0.03516377649325626, 0....",[],[],-6,0,np,North,11.75
3,4,False,False,False,True,False,False,1.000000,0.000000,32971.044566,...,"[2201.849, 2183.754, 2170.5735, 2184.0732, 199...","[399.80379953797336, 379.8746709442471, 393.98...","[0.07407407407407407, 0.07487401007919367, 0.0...",[],[],-6,0,np,North,12.75
4,5,False,False,False,True,False,False,1.000000,0.000000,68633.842773,...,"[762.9454, 715.47797, 717.8798, 737.74695, 697...","[369.1166185977001, 370.10013425423125, 351.96...","[0.16309291747888238, 0.13398692810457516, 0.1...",[],[],-6,0,np,North,3.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19680,1611,False,False,False,False,True,False,0.578435,0.689491,13579.133911,...,"[6442.857, 6306.4277, 6554.154, 6406.6074]","[398.28177839313713, 422.6973096969036, 414.11...","[0.0, 0.0, 0.0, 0.0]",[],[],-36,-30,sp,South,0.75
19681,1612,False,False,False,False,True,False,0.443903,0.638276,8719.815552,...,"[7184.1323, 7240.047, 7322.027, 7336.134]","[509.69726002304543, 907.6915431466794, 524.54...","[0.0, 0.017094017094017096, 0.0, 0.0]",[],[],-36,-30,sp,South,0.75
19682,1613,False,False,False,False,True,False,0.888643,0.398896,3336.406281,...,"[6847.8555, 6892.512, 6975.3403, 6954.3286]","[782.5978373340934, 587.838466332376, 764.4934...","[0.0, 0.0, 0.0, 0.0]",[],[],-36,-30,sp,South,0.75
19683,1614,False,False,False,True,False,False,1.000000,0.086406,1891.509369,...,"[6809.817, 6663.1562, 6599.3247, 6513.1787]","[522.8713991030681, 756.9745702465837, 567.517...","[0.0, 0.0, 0.0, 0.0]",[],[],-36,-30,sp,South,0.75


In [6]:
valid_clouds = cloud_data[cloud_data["val_intersec_lat"].map(lambda x: isinstance(x, np.ndarray) and len(x) > 0)]

In [7]:
test_row = valid_clouds.iloc[10]

In [8]:
test_row["avg_size[km]"]

3545.9735281808034

In [25]:
ALLOWED_CTH_DEV=1000

def check_val_seviri_coincidence(row):
    points_of_interest = row["val_cth_hist"] > 0
    if type(row["cth_hist"]) == float:
            N+=1
            return np.nan 
    cth_deviations = np.abs(row["cth_hist"][points_of_interest] - row["val_cth_hist"][points_of_interest])
    if cth_deviations.size == 0:
        return False
    mean_height_dev = cth_deviations.mean()  
    if mean_height_dev > ALLOWED_CTH_DEV:
        return False
    return True

def get_val_seviri_coincidence(row):
    if "val_seviri_coincidence" not in row.keys():
        return check_val_seviri_coincidence(row)
    else:
        return row["val_seviri_coincidence"]

def identify_times_to_process(row):
    val_if_hist = row.val_ice_frac_hist
    idx = np.where((val_if_hist >= -0.01) & (val_if_hist <= 1.01))[0]
    return row.track_start_time+idx*pd.Timedelta(minutes=15)

In [ ]:
cloud_data["val_seviri_coincidence"] = cloud_data.apply(get_val_seviri_coincidence,axis=1)
valid_clouds = cloud_data[cloud_data["val_seviri_coincidence"]==True]


In [26]:
times_to_process = pd.DatetimeIndex(
    np.concatenate(valid_clouds.apply(identify_times_to_process, axis=1).values)
).unique().sort_values()